# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.02 · Piloto opcional de longitud de chunks

Compara localmente ventanas de 15, 20, 25, 30 y 35 segundos mediante perfiles progresivos: smoke test, confirmación corta, perfil robusto de unos 30 minutos y diagnósticos neuronales secundarios.

La selección de hiperparámetros usa exclusivamente `validation`; consultar `test` para elegir introduciría sesgo de selección [1]. Se promedia *average precision* de los cuatro daños por ser una medida informativa ante desbalance [2]. ComplementNB y SGD con TF-IDF reutilizan el mismo entrenador de los cuadernos posteriores y la implementación de scikit-learn [3]. El comparador neuronal reutiliza como encoder congelado `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` [4] y aplica *mean pooling*; no equivale a un ajuste fino. `gemma3:4b`, el LLM de menor tamaño de archivo entre los tres modelos Ollama descargados, se limita a tres filas por longitud y salida estructurada [5] [6]. Sus etiquetas duras y la AP de MiniLM no son métricas intercambiables y no intervienen en la recomendación automática. El perfil robusto preserva cinco cohortes pareadas y remuestrea videos completos, no chunks como si fueran independientes, mediante bootstrap agrupado [7] [8]. La referencia de 30 s y el margen de no inferioridad de 0.01 se predeclaran antes de la corrida; esta regla evita escoger retrospectivamente la referencia. La transferencia de etiquetas por mayor solapamiento temporal, la muestra enriquecida, la tolerancia absoluta de 0.02 AP y el proxy de costo `filas_train × modelos` son decisiones metodológicas locales. El resultado es orientativo, no una estimación productiva; `test` se muestra solo después y nunca participa en la recomendación.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [1]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


raíz,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4
backend,local


## Controles opcionales y elección manual

In [2]:
RUN_CHUNK_LENGTH_SMOKE_TEST=False
RUN_CHUNK_LENGTH_CONFIRMATORY_TEST=False
RUN_CHUNK_LENGTH_ROBUST_TEST=True
RUN_BOUNDED_HF_COMPARISON=False
RUN_BOUNDED_OLLAMA_COMPARISON=False
CANDIDATE_SECONDS=(15,20,25,30,35)
TOY_MODELS=('complement_nb','sgd_incremental')
TOY_VIDEO_LIMITS={'train':40,'validation':16,'test':16}
TOY_MAX_FEATURES=12000
NEURAL_CANDIDATE_SECONDS=(20,30)
HF_SMOKE_MODEL='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
HF_SMOKE_REVISION='e8f8c211226b894fcb81acc59f3b34ba3efd5f42'
HF_SMOKE_TRAIN_LIMIT=120
HF_SMOKE_VALIDATION_LIMIT=40
HF_SMOKE_BATCH_SIZE=16
OLLAMA_SMOKE_MODEL='gemma3:4b'
OLLAMA_SMOKE_VALIDATION_LIMIT=3
OLLAMA_SMOKE_TIMEOUT_SECONDS=90.0
OLLAMA_SMOKE_MAX_WALL_SECONDS=600.0
CONFIRMATORY_MODELS=('complement_nb','logistic_regression','sgd_incremental')
CONFIRMATORY_VIDEO_LIMITS={'train':200,'validation':80,'test':80}
CONFIRMATORY_SEEDS=(20260805,20260817,20260829)
CONFIRMATORY_MAX_FEATURES=20000
ROBUST_VIDEO_LIMITS={'train':300,'validation':100,'test':100}
ROBUST_SEEDS=(20260805,20260817,20260829,20260841,20260853)
ROBUST_MAX_FEATURES=25000
ROBUST_REFERENCE_SECONDS=30.0
ROBUST_NONINFERIORITY_MARGIN=0.01
ROBUST_BOOTSTRAP_REPLICATES=1000
ROBUST_CONFIDENCE_LEVEL=0.95
ROBUST_BOOTSTRAP_SEED=20260807
ROBUST_RUNTIME_BUDGET_SECONDS=1800.0
MAX_VALIDATION_AP_DROP=0.02
MANUAL_CHUNK_SECONDS=30.0  # Puede elegirse cualquier valor positivo
USE_SMOKE_RECOMMENDATION=False
USE_CONFIRMATORY_RECOMMENDATION=False
USE_ROBUST_RECOMMENDATION=False
APPLY_CHUNK_SELECTION=False  # Si es False, no mueve ningún dataset
from moderacion_peru.colab import prepare_local_bundle_input
from moderacion_peru.chunk_optimization import activate_chunking_configuration, run_bounded_neural_chunk_comparison, run_chunk_length_confirmatory_test, run_chunk_length_robust_test, run_chunk_length_smoke_test
from moderacion_peru.incremental import DEFAULT_CHUNKING_CONFIGURATION
import json
TRANSCRIPTS=ROOT/'datos/raw/transcripts_raw.jsonl'
CHUNKS_CHECKPOINT=prepare_local_bundle_input('chunks_v2',project_root=ROOT)
CHUNKS=Path(CHUNKS_CHECKPOINT['path'])
DATASET_CHECKPOINT=prepare_local_bundle_input('dataset_5_salidas',project_root=ROOT)
DATASET=Path(DATASET_CHECKPOINT['path'])
PILOT_ROOT=ROOT/'resultados/pilotos/chunk_length'
ROBUST_ROOT=PILOT_ROOT/'robust_30min'
RECOMMENDATION=PILOT_ROOT/'recommendation.json'
CONFIRMATORY_RECOMMENDATION=PILOT_ROOT/'confirmatory_recommendation.json'
ROBUST_RECOMMENDATION=ROBUST_ROOT/'robust_recommendation.json'
if RUN_CHUNK_LENGTH_CONFIRMATORY_TEST and RUN_CHUNK_LENGTH_ROBUST_TEST:
    raise ValueError('El perfil robusto ya incluye la confirmación; active solo uno de los dos')
show_summary('Configuración de pruebas', {'humo_rápido':RUN_CHUNK_LENGTH_SMOKE_TEST,'minilm_acotado':RUN_BOUNDED_HF_COMPARISON,'gemma_acotado':RUN_BOUNDED_OLLAMA_COMPARISON,'confirmatoria_corta':RUN_CHUNK_LENGTH_CONFIRMATORY_TEST,'robusta_30_min':RUN_CHUNK_LENGTH_ROBUST_TEST,'longitudes':CANDIDATE_SECONDS,'longitudes_neuronales':NEURAL_CANDIDATE_SECONDS,'cohortes_robustas':len(ROBUST_SEEDS),'bootstrap_replicates':ROBUST_BOOTSTRAP_REPLICATES,'dataset':DATASET,'aplicar_selección':APPLY_CHUNK_SELECTION}, tone='neutral')

humo_rápido,No
minilm_acotado,No
gemma_acotado,No
confirmatoria_corta,No
robusta_30_min,Sí
longitudes,"Ver detalle[ 15, 20, 25, 30, 35 ]"
longitudes_neuronales,"Ver detalle[ 20, 30 ]"
cohortes_robustas,5
bootstrap_replicates,1000
dataset,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/model_ready/v2/dataset_5_salidas.jsonl
aplicar_selección,No


## Prueba de humo local de extremo a extremo

In [3]:
if RUN_CHUNK_LENGTH_SMOKE_TEST:
    smoke_result=run_chunk_length_smoke_test(TRANSCRIPTS,CHUNKS,DATASET,PILOT_ROOT,candidate_seconds=CANDIDATE_SECONDS,model_names=TOY_MODELS,video_limits=TOY_VIDEO_LIMITS,max_features=TOY_MAX_FEATURES,max_validation_ap_drop=MAX_VALIDATION_AP_DROP)
    show_result('Recomendación del piloto',smoke_result['recommendation'],tone='success')
    show_table('Comparación por longitud',smoke_result['comparisons'],max_rows=len(CANDIDATE_SECONDS))
else:
    show_callout('Piloto desactivado','Cambie RUN_CHUNK_LENGTH_SMOKE_TEST=True para entrenar diez baselines CPU pequeños. Los resultados se reanudan por firma.',tone='neutral')

## Comparación neuronal acotada y no selectiva

In [4]:
if RUN_BOUNDED_HF_COMPARISON or RUN_BOUNDED_OLLAMA_COMPARISON:
    missing_neural_seconds=set(NEURAL_CANDIDATE_SECONDS)-set(CANDIDATE_SECONDS)
    if missing_neural_seconds:
        raise ValueError(f'Incluya estas longitudes neuronales en CANDIDATE_SECONDS: {sorted(missing_neural_seconds)}')
    neural_result=run_bounded_neural_chunk_comparison(PILOT_ROOT,candidate_seconds=NEURAL_CANDIDATE_SECONDS,run_hf=RUN_BOUNDED_HF_COMPARISON,run_ollama=RUN_BOUNDED_OLLAMA_COMPARISON,hf_model_id=HF_SMOKE_MODEL,hf_revision=HF_SMOKE_REVISION,hf_train_limit=HF_SMOKE_TRAIN_LIMIT,hf_validation_limit=HF_SMOKE_VALIDATION_LIMIT,hf_batch_size=HF_SMOKE_BATCH_SIZE,ollama_model=OLLAMA_SMOKE_MODEL,ollama_validation_limit=OLLAMA_SMOKE_VALIDATION_LIMIT,ollama_timeout_seconds=OLLAMA_SMOKE_TIMEOUT_SECONDS,max_ollama_wall_seconds=OLLAMA_SMOKE_MAX_WALL_SECONDS)
    if 'huggingface' in neural_result:
        show_table('MiniLM congelado por longitud',neural_result['huggingface']['comparisons'],max_rows=len(NEURAL_CANDIDATE_SECONDS))
    if 'ollama' in neural_result:
        show_table('Gemma 3 4B: muestra descriptiva',neural_result['ollama']['comparisons'],max_rows=len(NEURAL_CANDIDATE_SECONDS))
    show_callout('Interpretación',neural_result['comparability_warning'],tone='warning')
else:
    show_callout('Comparación neuronal desactivada','Primero ejecute el smoke test CPU para materializar 20 s y 30 s. Luego active MiniLM, Gemma o ambos; Gemma procesa como máximo seis filas y dispone de un presupuesto total de diez minutos.',tone='neutral')

## Confirmación corta pareada

In [5]:
if RUN_CHUNK_LENGTH_CONFIRMATORY_TEST:
    confirmatory_result=run_chunk_length_confirmatory_test(TRANSCRIPTS,CHUNKS,DATASET,PILOT_ROOT,candidate_seconds=CANDIDATE_SECONDS,model_names=CONFIRMATORY_MODELS,video_limits=CONFIRMATORY_VIDEO_LIMITS,seeds=CONFIRMATORY_SEEDS,max_features=CONFIRMATORY_MAX_FEATURES)
    show_result('Recomendación confirmatoria',confirmatory_result['recommendation'],tone='success')
    show_table('Media y dispersión entre cohortes pareadas',confirmatory_result['aggregated_comparisons'],max_rows=len(CANDIDATE_SECONDS))
else:
    show_callout('Confirmación desactivada','Active RUN_CHUNK_LENGTH_CONFIRMATORY_TEST=True solo después del piloto rápido. Reentrena e infiere 45 baselines CPU: 5 longitudes × 3 modelos × 3 cohortes.',tone='neutral')

## Perfil robusto de aproximadamente 30 minutos

In [6]:
if RUN_CHUNK_LENGTH_ROBUST_TEST:
    robust_result=run_chunk_length_robust_test(TRANSCRIPTS,CHUNKS,DATASET,ROBUST_ROOT,candidate_seconds=CANDIDATE_SECONDS,reference_seconds=ROBUST_REFERENCE_SECONDS,model_names=CONFIRMATORY_MODELS,video_limits=ROBUST_VIDEO_LIMITS,seeds=ROBUST_SEEDS,max_features=ROBUST_MAX_FEATURES,bootstrap_replicates=ROBUST_BOOTSTRAP_REPLICATES,confidence_level=ROBUST_CONFIDENCE_LEVEL,noninferiority_margin=ROBUST_NONINFERIORITY_MARGIN,bootstrap_seed=ROBUST_BOOTSTRAP_SEED,runtime_budget_seconds=ROBUST_RUNTIME_BUDGET_SECONDS)
    show_result('Recomendación robusta',robust_result['recommendation'],tone='success')
    show_table('Bootstrap pareado por video',robust_result['bootstrap']['comparisons'],max_rows=len(CANDIDATE_SECONDS))
    show_summary('Tiempo y presupuesto',robust_result['runtime'],tone='warning' if robust_result['runtime']['exceeded_budget'] else 'success')
else:
    show_callout('Perfil robusto desactivado','Active RUN_CHUNK_LENGTH_ROBUST_TEST=True para 75 ajustes CPU y 1000 réplicas bootstrap agrupadas por video. Reanuda modelos por firma y apunta a unos 30 minutos.',tone='neutral')

schema_version,1.0
selection_version,1.2.0
profile,paired_video_cluster_bootstrap
recommended_seconds,30.0
reference_seconds,30.0
noninferiority_margin,0.01
paired_validation_ap_macro_damage,0.12327712156763271
delta_vs_reference,0.0
delta_vs_reference_ci_low,0.0
delta_vs_reference_ci_high,0.0
compute_proxy,43297


TypeError: show_table() got an unexpected keyword argument 'limit'

## Previsualización o activación reversible

In [ ]:
if USE_ROBUST_RECOMMENDATION:
    if not ROBUST_RECOMMENDATION.is_file():
        raise FileNotFoundError('Ejecute primero el perfil robusto o seleccione MANUAL_CHUNK_SECONDS')
    selected_seconds=float(json.loads(ROBUST_RECOMMENDATION.read_text(encoding='utf-8-sig'))['recommended_seconds'])
    selection_source='01_02_robust_bootstrap_recommendation'
elif USE_CONFIRMATORY_RECOMMENDATION:
    if not CONFIRMATORY_RECOMMENDATION.is_file():
        raise FileNotFoundError('Ejecute primero la confirmación corta o seleccione MANUAL_CHUNK_SECONDS')
    selected_seconds=float(json.loads(CONFIRMATORY_RECOMMENDATION.read_text(encoding='utf-8-sig'))['recommended_seconds'])
    selection_source='01_02_confirmatory_recommendation'
elif USE_SMOKE_RECOMMENDATION:
    if not RECOMMENDATION.is_file():
        raise FileNotFoundError('Ejecute primero el piloto o seleccione MANUAL_CHUNK_SECONDS')
    selected_seconds=float(json.loads(RECOMMENDATION.read_text(encoding='utf-8-sig'))['recommended_seconds'])
    selection_source='01_02_smoke_recommendation'
else:
    selected_seconds=float(MANUAL_CHUNK_SECONDS)
    selection_source='01_02_manual'
if selected_seconds <= 0:
    raise ValueError('MANUAL_CHUNK_SECONDS debe ser positivo')
selected_config={**DEFAULT_CHUNKING_CONFIGURATION,'max_seconds':selected_seconds}
if APPLY_CHUNK_SELECTION:
    activation=activate_chunking_configuration(ROOT,selected_config,source=selection_source)
    show_result('Configuración activada sin borrar derivados',activation,tone='success')
else:
    show_summary('Selección previsualizada',{'segundos':selected_seconds,'origen':selection_source,'acción':'Active APPLY_CHUNK_SELECTION=True; 01_03 materializará o restaurará esta firma.'},tone='neutral')

## Referencias

[1] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.

[2] T. Saito and M. Rehmsmeier, "The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets," PLOS ONE, vol. 10, no. 3, Art. no. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[3] F. Pedregosa, G. Varoquaux, A. Gramfort, et al., "Scikit-Learn: Machine Learning in Python," J. Mach. Learn. Res., vol. 12, pp. 2825–2830, 2011. [Online]. Available: https://www.jmlr.org/papers/v12/pedregosa11a.html

[4] Sentence Transformers, "Model Card: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2," Hugging Face Hub, revision e8f8c211226b894fcb81acc59f3b34ba3efd5f42, 2026. [Online]. Available: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/tree/e8f8c211226b894fcb81acc59f3b34ba3efd5f42

[5] Ollama, "Model Card: gemma3:4b," Ollama Model Library, 2026. [Online]. Available: https://ollama.com/library/gemma3:4b. Accessed: Aug. 6, 2026.

[6] Ollama, "Structured Outputs," Ollama Documentation, 2026. [Online]. Available: https://docs.ollama.com/capabilities/structured-outputs. Accessed: Aug. 5, 2026.

[7] B. Efron, "Bootstrap Methods: Another Look at the Jackknife," The Annals of Statistics, vol. 7, no. 1, pp. 1–26, 1979, doi: 10.1214/aos/1176344552.

[8] C. A. Field and A. H. Welsh, "Bootstrapping Clustered Data," Journal of the Royal Statistical Society: Series B, vol. 69, no. 3, pp. 369–390, 2007, doi: 10.1111/j.1467-9868.2007.00593.x.